<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Generator Audiovisual with Diffusers

This notebook runs Cosmos3 audiovisual generation directly with `Cosmos3OmniPipeline`.

Run all Cosmos3-Nano examples first, then run the Cosmos3-Super T2V/I2V examples without audio. Each section loads the matching model explicitly.

Note: if you have already completed steps 1-3 and installed the `Cosmos3 Diffusers (Python 3.13)` kernel, switch to that kernel and jump directly to step 4. Run the restore cell there, then continue with verification and the examples.


## 1. Prerequisites

Use a Linux machine with NVIDIA GPU access, model access on Hugging Face, and either `uvx hf@latest auth login` or `HF_TOKEN` set.

> **Headless servers:** if you see an error like `libxcb.so.1: cannot open shared object file` (a missing system graphics library) when importing or running the pipeline, install the required system libraries:
>
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

> **uv version:** these notebooks need `uv >= 0.11.3`. Older versions fail to parse the project config and do not recognize newer `--torch-backend` values such as `cu130` (you may see errors like `a value is required for '--torch-backend'` or an invalid-value list that stops at `cu129`). If you hit version-related errors, upgrade with `uv self update` (or reinstall from https://astral.sh/uv).


## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 or 12.8 Torch backend depending on the CUDA version installed on your system (`cu130` or `cu128`):

```bash
export COSMOS3_DIFFUSERS_VENV=/path/to/.venv-cosmos3-diffusers
export COSMOS3_TORCH_BACKEND=cu130
export HF_HOME=/path/to/large/huggingface/cache
export UV_LINK_MODE=copy
export CUDA_VISIBLE_DEVICES=0
```


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def configure_diffusers_environment() -> None:
    global COSMOS_ROOT
    global COSMOS3_AUDIOVISUAL_ROOT
    global COSMOS3_DIFFUSERS_VENV
    global COSMOS3_TORCH_BACKEND
    global COSMOS3_AUDIOVISUAL_OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
    COSMOS3_DIFFUSERS_VENV = Path(
        os.environ.get("COSMOS3_DIFFUSERS_VENV", COSMOS_ROOT / ".venv-cosmos3-diffusers")
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get("COSMOS3_TORCH_BACKEND", "cu130")
    COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
        os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
    ).resolve()

    os.environ["COSMOS3_DIFFUSERS_VENV"] = str(COSMOS3_DIFFUSERS_VENV)
    os.environ["COSMOS3_TORCH_BACKEND"] = COSMOS3_TORCH_BACKEND
    os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
    os.environ.setdefault("UV_CACHE_DIR", str(Path.home() / ".cache" / "uv"))
    os.environ.setdefault("UV_LINK_MODE", "copy")
    os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

    print(f"COSMOS_ROOT: {COSMOS_ROOT}")
    for key in [
        "COSMOS3_DIFFUSERS_VENV",
        "COSMOS3_TORCH_BACKEND",
        "COSMOS3_AUDIOVISUAL_OUTPUT_ROOT",
        "UV_CACHE_DIR",
        "UV_LINK_MODE",
        "HF_HOME",
        "HF_HUB_DISABLE_XET",
        "CUDA_VISIBLE_DEVICES",
    ]:
        print(f"{key}: {os.environ[key]}")
    print("HF_TOKEN:", "<set>" if os.environ.get("HF_TOKEN") else "<unset>")


configure_diffusers_environment()


## 3. Install Diffusers Dependencies


In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export UV_LINK_MODE="${UV_LINK_MODE:-copy}"
uv venv "$COSMOS3_DIFFUSERS_VENV" --python 3.13 --seed --managed-python --allow-existing
source "$COSMOS3_DIFFUSERS_VENV/bin/activate"
uv pip install --torch-backend="$COSMOS3_TORCH_BACKEND" \
  "diffusers @ git+https://github.com/huggingface/diffusers.git" \
  accelerate \
  av \
  cosmos_guardrail \
  huggingface_hub \
  imageio \
  imageio-ffmpeg \
  ipykernel \
  torch \
  torchvision \
  transformers

"$COSMOS3_DIFFUSERS_VENV/bin/python" -m ipykernel install --user \
  --name cosmos3-diffusers \
  --display-name "Cosmos3 Diffusers (Python 3.13)"

echo
echo "Installed dependencies into: $COSMOS3_DIFFUSERS_VENV"
echo "Next: switch this notebook kernel to: Cosmos3 Diffusers (Python 3.13)"
echo "After switching kernels, run the Restore Environment cell below, then continue with Verify."


## 4. Select the Diffusers Kernel

The install cell creates and registers the `Cosmos3 Diffusers (Python 3.13)` Jupyter kernel. 

**Note**: Switch this notebook to that kernel before running the remaining Python cells, then run the restore cell immediately below. It can take some time for the new Jupyter kernel to show up in the notebook interface.

In [ ]:
# Run this cell immediately after switching to the Cosmos3 Diffusers kernel.
# It restores the same paths and cache settings as the setup cell above.
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def configure_diffusers_environment() -> None:
    global COSMOS_ROOT
    global COSMOS3_AUDIOVISUAL_ROOT
    global COSMOS3_DIFFUSERS_VENV
    global COSMOS3_TORCH_BACKEND
    global COSMOS3_AUDIOVISUAL_OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
    COSMOS3_DIFFUSERS_VENV = Path(
        os.environ.get("COSMOS3_DIFFUSERS_VENV", COSMOS_ROOT / ".venv-cosmos3-diffusers")
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get("COSMOS3_TORCH_BACKEND", "cu130")
    COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
        os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
    ).resolve()

    os.environ["COSMOS3_DIFFUSERS_VENV"] = str(COSMOS3_DIFFUSERS_VENV)
    os.environ["COSMOS3_TORCH_BACKEND"] = COSMOS3_TORCH_BACKEND
    os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
    os.environ.setdefault("UV_CACHE_DIR", str(Path.home() / ".cache" / "uv"))
    os.environ.setdefault("UV_LINK_MODE", "copy")
    os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

    print(f"COSMOS_ROOT: {COSMOS_ROOT}")
    for key in [
        "COSMOS3_DIFFUSERS_VENV",
        "COSMOS3_TORCH_BACKEND",
        "COSMOS3_AUDIOVISUAL_OUTPUT_ROOT",
        "UV_CACHE_DIR",
        "UV_LINK_MODE",
        "HF_HOME",
        "HF_HUB_DISABLE_XET",
        "CUDA_VISIBLE_DEVICES",
    ]:
        print(f"{key}: {os.environ[key]}")
    print("HF_TOKEN:", "<set>" if os.environ.get("HF_TOKEN") else "<unset>")


configure_diffusers_environment()


## 5. Verify GPU and Python Environment


In [ ]:
import os
import sys
from pathlib import Path

if "COSMOS3_DIFFUSERS_VENV" not in os.environ:
    raise RuntimeError("Run the Restore Environment cell after switching to the Diffusers kernel.")

expected_python = (Path(os.environ["COSMOS3_DIFFUSERS_VENV"]) / "bin" / "python").resolve()
current_python = Path(sys.executable).resolve()
print("kernel python:", current_python)
print("expected python:", expected_python)
if current_python != expected_python:
    raise RuntimeError(
        "This notebook is not running inside the Diffusers venv. "
        "Switch the notebook kernel to 'Cosmos3 Diffusers (Python 3.13)', then run the Restore Environment cell above."
    )

import torch
import diffusers

print("diffusers:", diffusers.__version__)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))


## 6. Preview Available Inputs


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

assets_dir = COSMOS3_AUDIOVISUAL_ROOT / "assets"
for prompt_dir in sorted((assets_dir / "prompts").iterdir()):
    if not prompt_dir.is_dir():
        continue
    print(f"{prompt_dir.relative_to(assets_dir)}:")
    for prompt_path in sorted(prompt_dir.glob("*.json")):
        data = json.loads(prompt_path.read_text())
        caption = (
            data.get("temporal_caption")
            or data.get("comprehensive_t2i_caption")
            or data.get("extra", {}).get("prompt", "")
        )
        print(f"  {prompt_path.name}: {caption[:180]}{'...' if len(caption) > 180 else ''}")
    print()

for image_dir in sorted((assets_dir / "images").iterdir()):
    if not image_dir.is_dir():
        continue
    print(f"{image_dir.relative_to(assets_dir)}:")
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            print(f"  {image_path.name}")
            display(Image(filename=str(image_path), width=420))


## 7. Define Asset Sets, Payload Helpers, Runner, and Viewer Helpers


In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FIXED_SAMPLING = {
    "num_steps": 35,
    "guidance": 6.0,
    "shift": 10.0,
    "fps": 24,
    "num_frames": 189,
    "resolution": "720",
    "aspect_ratio": "16,9",
    "seed": 1234,
}

# All asset paths are repo-relative under cookbooks/cosmos3/generator/audiovisual.
# The Nano/Super cases without audio are intentionally separate so each run uses the matching model.
ASSET_SETS = {
    "t2i": {
        "model": "Cosmos3-Nano",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2i_super": {
        "model": "Cosmos3-Super",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "enable_sound": False,
    },
    "t2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "t2vs": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_pouring_water_audio.json",
        "enable_sound": True,
    },
    "i2v_nano_noaudio": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
    "i2vs": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/coastal_road_audio.json",
        "image": "assets/images/image2video/coastal_road_audio.jpg",
        "enable_sound": True,
    },
    "t2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "enable_sound": False,
    },
    "i2v_super_noaudio": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "enable_sound": False,
    },
}


def asset_path(relative_path: str) -> Path:
    path = COSMOS3_AUDIOVISUAL_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


def compact_json_file(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def payload_dimensions(payload: dict) -> tuple[int, int]:
    if payload.get("resolution") == "720" and payload.get("aspect_ratio") == "16,9":
        return 720, 1280
    if payload.get("resolution") == "256" and payload.get("aspect_ratio") == "16,9":
        return 192, 320
    raise ValueError(f"Unsupported payload resolution/aspect ratio: {payload.get('resolution')} {payload.get('aspect_ratio')}")


def resolve_payload_path(payload_path: Path, value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return (payload_path.parent / path).resolve()


def create_payload(use_case: str, *, backend: str) -> tuple[Path, Path, str]:
    spec = ASSET_SETS[use_case]
    payload_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / "payloads" / use_case
    output_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / use_case
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = asset_path(spec["prompt"])
    negative_prompt = ""
    if spec["mode"] != "text2image":
        negative_prompt_path = asset_path(f"assets/negative_prompts/{spec['mode']}/neg_prompt.json")
        negative_prompt = compact_json_file(negative_prompt_path)
    payload_path = payload_dir / f"{use_case}.json"
    payload = {
        "model_mode": spec["mode"],
        "name": use_case,
        "prompt": compact_json_file(prompt_path),
        "negative_prompt": negative_prompt,
        "enable_sound": spec["enable_sound"],
        **FIXED_SAMPLING,
    }
    if spec["mode"] == "image2video":
        image_path = asset_path(spec["image"])
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")

    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_INPUT"] = str(payload_path)
    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_OUTPUT"] = str(output_dir)

    print(f"model:   {spec['model']}")
    print(f"payload: {payload_path}")
    print(f"output:  {output_dir}")
    print(f"prompt:  {prompt_path.relative_to(COSMOS_ROOT)}")
    if "vision_path" in payload:
        image_display_path = resolve_payload_path(payload_path, payload["vision_path"])
        print(f"image:   {image_display_path.relative_to(COSMOS_ROOT)}")
        display(Image(filename=str(image_display_path), width=420))
    print(json.dumps({k: payload[k] for k in ["model_mode", "name", "enable_sound", "num_steps", "guidance", "shift", "fps", "num_frames", "resolution", "aspect_ratio", "seed"]}, indent=2))
    return payload_path, output_dir, spec["model"]


import json
import gc
import os
import time
from pathlib import Path

import sys

if "COSMOS3_DIFFUSERS_VENV" not in os.environ:
    raise RuntimeError("Run the Restore Environment cell after switching to the Diffusers kernel.")
expected_python = (Path(os.environ["COSMOS3_DIFFUSERS_VENV"]) / "bin" / "python").resolve()
if Path(sys.executable).resolve() != expected_python:
    raise RuntimeError("Switch the notebook kernel to 'Cosmos3 Diffusers (Python 3.13)' before running Diffusers cells.")

import torch
from diffusers import Cosmos3OmniPipeline
from diffusers import logging as diffusers_logging
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import encode_video, export_to_video, load_image

MODEL_IDS = {
    "Cosmos3-Nano": "nvidia/Cosmos3-Nano",
    "Cosmos3-Super": "nvidia/Cosmos3-Super",
}

_pipe = None
_pipe_model = None


def resolve_model_id(model: str) -> str:
    return MODEL_IDS.get(model, model)


def get_pipe(model: str) -> Cosmos3OmniPipeline:
    global _pipe, _pipe_model
    model_id = resolve_model_id(model)
    if _pipe is not None and _pipe_model == model_id:
        return _pipe
    if _pipe is not None:
        del _pipe
        _pipe = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    diffusers_logging.set_verbosity_info()
    print(f"loading {model_id}...")
    t0 = time.time()
    pipe = Cosmos3OmniPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        safety_checker=None,
        enable_safety_checker=True,
        token=os.environ.get("HF_TOKEN") or None,
    )
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FIXED_SAMPLING["shift"])
    pipe.to("cuda")
    _pipe = pipe
    _pipe_model = model_id
    print(f"loaded pipeline in {time.time() - t0:.1f}s")
    return _pipe


def run_diffusers_payload(payload_path: Path, output_dir: str | Path, *, model: str) -> Path:
    payload_path = Path(payload_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = json.loads(payload_path.read_text())
    height, width = payload_dimensions(payload)
    pipe = get_pipe(model)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=payload["shift"])
    generator = torch.Generator(device="cuda").manual_seed(payload["seed"])
    image = load_image(str(resolve_payload_path(payload_path, payload["vision_path"]))) if payload["model_mode"] == "image2video" else None
    output_ext = ".png" if payload["model_mode"] == "text2image" else ".mp4"
    output_path = output_dir / f"{payload['name']}{output_ext}"

    print(f"generating {payload['model_mode']} {payload['name']} with {model} -> {output_path}")
    t0 = time.time()
    if payload["model_mode"] == "text2image":
        result = pipe(
            prompt=payload["prompt"],
            negative_prompt="",
            num_frames=1,
            height=height,
            width=width,
            num_inference_steps=payload["num_steps"],
            guidance_scale=payload["guidance"],
            add_resolution_template=False,
            add_duration_template=False,
            generator=generator,
        )
        print(f"generated in {time.time() - t0:.1f}s")
        result.video[0].save(output_path)
        print(f"wrote {output_path}")
        return output_path

    result = pipe(
        prompt=payload["prompt"],
        negative_prompt=payload["negative_prompt"],
        image=image,
        num_frames=payload["num_frames"],
        height=height,
        width=width,
        fps=payload["fps"],
        num_inference_steps=payload["num_steps"],
        guidance_scale=payload["guidance"],
        enable_sound=payload["enable_sound"],
        add_resolution_template=False,
        add_duration_template=False,
        generator=generator,
    )
    print(f"generated in {time.time() - t0:.1f}s")
    if payload["enable_sound"] and result.sound is not None:
        encode_video(
            result.video,
            fps=payload["fps"],
            output_path=str(output_path),
            audio=result.sound,
            audio_sample_rate=pipe.sound_tokenizer.config.sampling_rate,
        )
    else:
        export_to_video(result.video, str(output_path), fps=payload["fps"], macro_block_size=1)
    print(f"wrote {output_path}")
    return output_path


import base64
import html
from pathlib import Path
from IPython.display import HTML, display


def display_video(path: Path, *, width: int = 720) -> None:
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(str(path))
    markup = f"""
<video controls playsinline preload="metadata" width="{width}" style="max-width: 100%; background: #000;">
  <source src="data:video/mp4;base64,{data}" type="video/mp4">
</video>
<div style="font-family: monospace; font-size: 12px; margin-top: 4px;">{label}</div>
"""
    display(HTML(markup))


def view_run(output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    videos = [
        path
        for path in sorted(output_dir.rglob("*.mp4"))
        if not path.name.endswith(("_preview.mp4", "_browser.mp4"))
    ]
    images = sorted(output_dir.rglob("*.png"))
    if not videos and not images:
        print(f"No generated media found under {output_dir}")
        return
    for src in videos:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display_video(src)
    for src in images:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display(Image(filename=str(src), width=720))


## Use Cases

Run each use case top-to-bottom: create the JSON payload, run inference, then view the generated media. Nano examples come first; Super examples are last.


## Nano: Text to Image

Nano text-to-image generation using a structured JSON prompt.

### Create Payload

In [ ]:
t2i_payload, t2i_output, t2i_model = create_payload("t2i", backend="diffusers")

### Run

In [ ]:
run_diffusers_payload(t2i_payload, t2i_output, model="Cosmos3-Nano")

### View Results

In [ ]:
view_run(t2i_output)

## Super: Text to Image

Super text-to-image generation using the same structured JSON prompt.

### Create Payload

In [ ]:
t2i_super_payload, t2i_super_output, t2i_super_model = create_payload("t2i_super", backend="diffusers")

### Run

In [ ]:
run_diffusers_payload(t2i_super_payload, t2i_super_output, model="Cosmos3-Super")

### View Results

In [ ]:
view_run(t2i_super_output)

## Nano: Text to Video Without Audio

Nano text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_nano_noaudio_payload, t2v_nano_noaudio_output, t2v_nano_noaudio_model = create_payload("t2v_nano_noaudio", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(t2v_nano_noaudio_payload, t2v_nano_noaudio_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(t2v_nano_noaudio_output)


## Nano: Text to Video with Audio

Nano text-to-video generation with generated audio.

### Create Payload


In [ ]:
t2vs_payload, t2vs_output, t2vs_model = create_payload("t2vs", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(t2vs_payload, t2vs_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(t2vs_output)


## Nano: Image to Video Without Audio

Nano image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_nano_noaudio_payload, i2v_nano_noaudio_output, i2v_nano_noaudio_model = create_payload("i2v_nano_noaudio", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(i2v_nano_noaudio_payload, i2v_nano_noaudio_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(i2v_nano_noaudio_output)


## Nano: Image to Video with Audio

Nano image-to-video generation using its paired image asset and generated audio.

### Create Payload


In [ ]:
i2vs_payload, i2vs_output, i2vs_model = create_payload("i2vs", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(i2vs_payload, i2vs_output, model="Cosmos3-Nano")


### View Results


In [ ]:
view_run(i2vs_output)


## Super: Text to Video Without Audio

Super text-to-video generation with audio disabled.

### Create Payload


In [ ]:
t2v_super_noaudio_payload, t2v_super_noaudio_output, t2v_super_noaudio_model = create_payload("t2v_super_noaudio", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(t2v_super_noaudio_payload, t2v_super_noaudio_output, model="Cosmos3-Super")


### View Results


In [ ]:
view_run(t2v_super_noaudio_output)


## Super: Image to Video Without Audio

Super image-to-video generation using its paired image asset, with audio disabled.

### Create Payload


In [ ]:
i2v_super_noaudio_payload, i2v_super_noaudio_output, i2v_super_noaudio_model = create_payload("i2v_super_noaudio", backend="diffusers")


### Run


In [ ]:
run_diffusers_payload(i2v_super_noaudio_payload, i2v_super_noaudio_output, model="Cosmos3-Super")


### View Results


In [ ]:
view_run(i2v_super_noaudio_output)
